In [ ]:
import os
import nibabel as nib
import numpy as np

from scipy.ndimage import zoom

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

#This is the path where the extract raw training data resides
ROOT = "/workspace/data/HECKTOR"

#This is the path where you want to save the Preprocessed .npz files
DESTINATION = "/workspace/data/Preprocessed"

TARGET_SPACING = np.array([1.0, 1.0, 1.0])

CUBE_SIZE = np.array([224, 224, 224])

PRIORS = {
    "HEAD_NECK": {
        "x": 0.50,
        "y": 0.42,
        "z": 0.48
    },

    "FULL_BODY": {
        "x": 0.50,
        "y": 0.42,
        "z": 0.82
    }
}

os.makedirs(DESTINATION, exist_ok=True)


In [ ]:
# ============================================================
# SCAN TYPE CLASSIFIER
# ============================================================

  def classify_scan(size_z_mm):

      if size_z_mm >= 700:
          return "FULL_BODY"

      return "HEAD_NECK"

In [ ]:
# ============================================================
# FIXED CUBE BOUNDS
# ============================================================

def get_cube_bounds(shape, center_xyz, cube_size):

    center_xyz = np.array(center_xyz).astype(int)

    start = center_xyz - cube_size // 2
    end = start + cube_size

    for d in range(3):

        if start[d] < 0:
            end[d] -= start[d]
            start[d] = 0

        if end[d] > shape[d]:

            shift = end[d] - shape[d]

            start[d] -= shift
            end[d] = shape[d]

        start[d] = max(start[d], 0)

    return start, end

In [ ]:
# ============================================================
# PAD TO TARGET SHAPE
# ============================================================

def pad_to_shape(arr, target_shape, pad_value=0):

    pads = []

    for current, target in zip(arr.shape, target_shape):

        total_pad = max(0, target - current)

        before = total_pad // 2
        after = total_pad - before

        pads.append((before, after))

    return np.pad(
        arr,
        pads,
        mode="constant",
        constant_values=pad_value
    )

In [ ]:
# ============================================================
# MAIN LOOP
# ============================================================

patients = sorted(os.listdir(ROOT))

patient_no = 0

for p in patients:

    patient_dir = os.path.join(ROOT, p)

    if not os.path.isdir(patient_dir):
        continue

    patient_no += 1

    print(f"\nProcessing patient {patient_no}: {p}")

    ct_path = os.path.join(
        patient_dir,
        f"{p}__CT.nii.gz"
    )

    pt_path = os.path.join(
        patient_dir,
        f"{p}__PT.nii.gz"
    )

    mask_path = os.path.join(
        patient_dir,
        f"{p}.nii.gz"
    )

    if not (
        os.path.exists(ct_path)
        and os.path.exists(pt_path)
        and os.path.exists(mask_path)
    ):
        print("Missing files")
        continue

    try:

        # ====================================================
        # LOAD
        # ====================================================

        ct_img = nib.load(ct_path)
        pt_img = nib.load(pt_path)
        mask_img = nib.load(mask_path)

        ct = ct_img.get_fdata().astype(np.float32)
        pt = pt_img.get_fdata().astype(np.float32)
        mask = mask_img.get_fdata().astype(np.uint8)

        # ====================================================
        # SANITY CHECKS
        # ====================================================

        assert ct.shape == pt.shape == mask.shape

        ct_spacing = np.array(
            ct_img.header.get_zooms()[:3]
        )

        pt_spacing = np.array(
            pt_img.header.get_zooms()[:3]
        )

        mask_spacing = np.array(
            mask_img.header.get_zooms()[:3]
        )

        assert np.allclose(ct_spacing, pt_spacing)
        assert np.allclose(ct_spacing, mask_spacing)

        # ====================================================
        # SCAN TYPE
        # ====================================================

        size_z_mm = (
            ct.shape[2]
            * ct_spacing[2]
        )

        scan_type = classify_scan(
            size_z_mm
        )

        print(f"Scan type: {scan_type}")

        # ====================================================
        # RESAMPLE
        # ====================================================

        zoom_factor = (
            ct_spacing
            / TARGET_SPACING
        )

        ct_res = zoom(
            ct,
            zoom_factor,
            order=1
        ).astype(np.float32)

        pt_res = zoom(
            pt,
            zoom_factor,
            order=1
        ).astype(np.float32)

        mask_res = zoom(
            mask,
            zoom_factor,
            order=0
        ).astype(np.uint8)

        assert (
            ct_res.shape
            == pt_res.shape
            == mask_res.shape
        )

        print(
            f"Resampled shape: "
            f"{ct_res.shape}"
        )

        # ====================================================
        # ANATOMICAL PRIOR CENTER
        # ====================================================

        sx, sy, sz = ct_res.shape

        center_x = int(
            PRIORS[scan_type]["x"] * sx
        )

        center_y = int(
            PRIORS[scan_type]["y"] * sy
        )

        center_z = int(
            PRIORS[scan_type]["z"] * sz
        )

        start, end = get_cube_bounds(
            ct_res.shape,
            (
                center_x,
                center_y,
                center_z
            ),
            CUBE_SIZE
        )

        # ====================================================
        # CROP
        # ====================================================

        ct_crop = ct_res[
            start[0]:end[0],
            start[1]:end[1],
            start[2]:end[2]
        ]

        pt_crop = pt_res[
            start[0]:end[0],
            start[1]:end[1],
            start[2]:end[2]
        ]

        mask_crop = mask_res[
            start[0]:end[0],
            start[1]:end[1],
            start[2]:end[2]
        ]

        # ====================================================
        # PAD IF NECESSARY
        # ====================================================

        ct_crop = pad_to_shape(
            ct_crop,
            CUBE_SIZE,
            pad_value=-1024
        )

        pt_crop = pad_to_shape(
            pt_crop,
            CUBE_SIZE,
            pad_value=0
        )

        mask_crop = pad_to_shape(
            mask_crop,
            CUBE_SIZE,
            pad_value=0
        )

        assert ct_crop.shape == tuple(CUBE_SIZE)
        assert pt_crop.shape == tuple(CUBE_SIZE)
        assert mask_crop.shape == tuple(CUBE_SIZE)

        # ====================================================
        # CT NORMALIZATION
        # ====================================================

        ct_crop = np.clip(
            ct_crop,
            -1024,
            1024
        )

        ct_crop = (
            ct_crop / 1024.0
        )

        # ====================================================
        # PET NORMALIZATION
        # ====================================================

        pt_mean = np.mean(pt_crop)
        pt_std = np.std(pt_crop)

        pt_crop = (
            pt_crop - pt_mean
        ) / (pt_std + 1e-8)

        # ====================================================
        # SAVE
        # ====================================================

        save_path = os.path.join(
            DESTINATION,
            f"{p}.npz"
        )

        np.savez_compressed(
            save_path,
            CT=ct_crop.astype(np.float32),
            PET=pt_crop.astype(np.float32),
            MASK=mask_crop.astype(np.uint8)
        )

    except Exception as e:

        print(
            f"Failed on {p}"
        )

        print(e)

<h1>Debug the Saved .npz Files</h1>

In [ ]:

# --------------------------------------------------------
# CONFIG
# --------------------------------------------------------

debug_path = destination_path

files = sorted([
    f for f in os.listdir(debug_path)
    if f.endswith(".npz")
])

print(f"Found {len(files)} npz files")

#files = files[778:]

print(f"Debugging the last {len(files)} npz files")

# --------------------------------------------------------
# DEBUG LOOP
# --------------------------------------------------------

for f in files[:4]:   # inspect first 4 files, you may change it if you want

    print("\n" + "="*60)
    print(f"DEBUGGING: {f}")
    print("="*60)

    file_path = os.path.join(debug_path, f)

    data = np.load(file_path)

    # ----------------------------------------------------
    # KEYS
    # ----------------------------------------------------

    print("\nKeys:")
    print(data.files)

    # ----------------------------------------------------
    # LOAD ARRAYS
    # ----------------------------------------------------

    ct = data["CT"]
    pet = data["PET"]
    mask = data["MASK"]

    # ----------------------------------------------------
    # SHAPES
    # ----------------------------------------------------

    print("\nShapes:")
    print("CT   :", ct.shape)
    print("PET  :", pet.shape)
    print("MASK :", mask.shape)

    # ----------------------------------------------------
    # DTYPES
    # ----------------------------------------------------

    print("\nDtypes:")
    print("CT   :", ct.dtype)
    print("PET  :", pet.dtype)
    print("MASK :", mask.dtype)

    # ----------------------------------------------------
    # CT STATS
    # ----------------------------------------------------

    print("\nCT Statistics:")
    print("Min  :", np.min(ct))
    print("Max  :", np.max(ct))
    print("Mean :", np.mean(ct))
    print("Std  :", np.std(ct))

    # ----------------------------------------------------
    # PET STATS
    # ----------------------------------------------------

    print("\nPET Statistics:")
    print("Min  :", np.min(pet))
    print("Max  :", np.max(pet))
    print("Mean :", np.mean(pet))
    print("Std  :", np.std(pet))

    # ----------------------------------------------------
    # MASK STATS
    # ----------------------------------------------------

    print("\nMask Statistics:")

    unique_labels = np.unique(mask)

    print("Unique labels:", unique_labels)

    for label in unique_labels:
        count = np.sum(mask == label)
        print(f"Label {label}: {count} voxels")

    # ----------------------------------------------------
    # SANITY CHECKS
    # ----------------------------------------------------

    print("\nSanity Checks:")

    assert ct.shape == (224, 224, 224)
    assert pet.shape == (224, 224, 224)
    assert mask.shape == (224, 224, 224)

    assert ct.dtype == np.float32
    assert pet.dtype == np.float32
    assert mask.dtype == np.uint8

    assert np.min(ct) >= -1.01
    assert np.max(ct) <= 1.01

    assert np.isfinite(ct).all()
    assert np.isfinite(pet).all()

    assert len(unique_labels) > 1

    print("All sanity checks PASSED")

Found 782 npz files
Debugging the last 4 npz files

DEBUGGING: USZ-008.npz

Keys:
['CT', 'PET', 'MASK']

Shapes:
CT   : (160, 160, 160)
PET  : (160, 160, 160)
MASK : (160, 160, 160)

Dtypes:
CT   : float32
PET  : float32
MASK : uint8

CT Statistics:
Min  : -1.0
Max  : 1.0
Mean : -0.29986724
Std  : 0.52527857

PET Statistics:
Min  : -0.69383824
Max  : 8.064413
Mean : 6.103516e-08
Std  : 0.99999994

Mask Statistics:
Unique labels: [0 1 2]
Label 0: 4054753 voxels
Label 1: 3335 voxels
Label 2: 37912 voxels

Sanity Checks:
All sanity checks PASSED

DEBUGGING: USZ-009.npz

Keys:
['CT', 'PET', 'MASK']

Shapes:
CT   : (160, 160, 160)
PET  : (160, 160, 160)
MASK : (160, 160, 160)

Dtypes:
CT   : float32
PET  : float32
MASK : uint8

CT Statistics:
Min  : -1.0
Max  : 1.0
Mean : -0.44890195
Std  : 0.53946316

PET Statistics:
Min  : -0.7555178
Max  : 8.335067
Mean : 2.746582e-07
Std  : 1.0

Mask Statistics:
Unique labels: [0 1 2]
Label 0: 4079930 voxels
Label 1: 6998 voxels
Label 2: 9072 voxels

Sa